# BirdCLEF 2026 — CPU Inference (TorchScript, no external deps)

**Competition constraints**: CPU only · no internet · 90-min budget

## What to upload to Kaggle before running

| File | How | Used as |
|---|---|---|
| `checkpoints/ensemble_baseline_panns.pt` | Datasets → New Dataset | weights (fast, recommended) |

The traced ensemble needs **no timm, no transformers, no src/** at runtime.
Only standard PyTorch + torchaudio (both pre-installed on Kaggle) are needed.

## Generating the ensemble locally

```bash
# Fast 2-model CNN ensemble (EfficientNet-B3 + PANNs) — use this for submission
python src/ensemble_export.py \
    --checkpoints checkpoints/exp001_baseline/best.pt \
                  checkpoints/exp005_panns/best.pt \
    --weights 0.45 0.55 \
    --output checkpoints/ensemble_baseline_panns.pt
```

> **Do not use `ensemble_v2_no_tta.pt`** (includes AST transformer, ~550 ms/window on CPU — will exceed the 90-min budget).

In [ ]:
# ── EDIT THIS ─────────────────────────────────────────────────────────────────
WEIGHTS_DATASET_SLUG = "your-weights-dataset-slug"   # slug for the dataset containing the .pt file
TRACED_FILENAME      = "ensemble_baseline_panns.pt"   # ← fast 2-model CNN ensemble (~100 ms/window)
#                      "ensemble_v2_no_tta.pt"        # ← includes AST (~550 ms/window) — will timeout
BATCH_SIZE           = 64    # windows per forward pass (larger = faster on CPU)
USE_TTA              = False
# ─────────────────────────────────────────────────────────────────────────────

import os, sys, time, math, csv
import numpy as np
import pandas as pd
import torch
import torchaudio.transforms as T
from pathlib import Path

TRACED_PATH = f"/kaggle/input/{WEIGHTS_DATASET_SLUG}/{TRACED_FILENAME}"
COMP_DIR    = "/kaggle/input/birdclef-2026"
OUT_CSV     = "/kaggle/working/submission.csv"

assert os.path.isfile(TRACED_PATH), f"Traced model not found: {TRACED_PATH}"
assert os.path.isdir(COMP_DIR),     f"Competition data not found: {COMP_DIR}"
print(f"Model : {TRACED_PATH}  ({os.path.getsize(TRACED_PATH)/1e6:.1f} MB)")
print(f"Output: {OUT_CSV}")

In [ ]:
# ── Audio helpers ──────────────────────────────────────────────────────────────
SAMPLE_RATE  = 32_000
CLIP_SAMPLES = 160_000   # 5 s × 32 kHz
N_MELS       = 128
N_FFT        = 1024
HOP_LENGTH   = 320
F_MIN        = 20.0
F_MAX        = 16_000.0

_MEL = T.MelSpectrogram(
    sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH,
    n_mels=N_MELS, f_min=F_MIN, f_max=F_MAX, power=2.0, center=False,
)
_DB = T.AmplitudeToDB(stype="power", top_db=80.0)


def load_waveform(path: str) -> torch.Tensor:
    """Load OGG → mono (1, T) at SAMPLE_RATE."""
    try:
        import soundfile as sf
        data, sr = sf.read(path, dtype="float32", always_2d=True)
        wav = torch.from_numpy(data.T)
    except Exception:
        import librosa
        data, sr = librosa.load(path, sr=None, mono=False)
        if data.ndim == 1:
            data = data[np.newaxis, :]
        wav = torch.from_numpy(data.astype(np.float32))
    if wav.shape[0] > 1:
        wav = wav.mean(0, keepdim=True)
    if sr != SAMPLE_RATE:
        import torchaudio.functional as F_a
        wav = F_a.resample(wav, sr, SAMPLE_RATE)
    return wav  # (1, T)


def file_to_specs(path: str):
    """Load file and compute ALL window mel-spectrograms in one batched STFT call.

    Returns (end_secs, specs) where specs is (W, 1, N_MELS, frames).
    Replaces the old per-window sliding_windows approach which called _MEL W times.
    """
    wav   = load_waveform(path)   # (1, T)
    total = wav.shape[-1]
    starts = list(range(0, total - CLIP_SAMPLES + 1, CLIP_SAMPLES))
    if not starts:
        return [], None

    # Stack raw audio slices: (W, CLIP_SAMPLES)
    chunks = torch.stack([wav[0, s : s + CLIP_SAMPLES] for s in starts])

    # One batched mel call instead of W separate calls: (W, N_MELS, frames)
    mel = _DB(_MEL(chunks))

    # Per-window normalisation (mean/std over freq+time for each window)
    mean = mel.mean(dim=(-2, -1), keepdim=True)
    std  = mel.std(dim=(-2, -1), keepdim=True)
    mel  = (mel - mean) / (std + 1e-6)

    specs    = mel.unsqueeze(1)  # (W, 1, N_MELS, frames)
    end_secs = [(s + CLIP_SAMPLES) // SAMPLE_RATE for s in starts]
    return end_secs, specs

print("Audio helpers ready")

In [ ]:
# ── Class list (must match training order) ─────────────────────────────────────
with open(os.path.join(COMP_DIR, "taxonomy.csv")) as f:
    ALL_CLASSES = [row["primary_label"] for row in csv.DictReader(f)]
NUM_CLASSES = len(ALL_CLASSES)
print(f"Classes: {NUM_CLASSES}")

In [ ]:
# ── Load traced model ──────────────────────────────────────────────────────────
device = torch.device("cpu")
torch.set_num_threads(os.cpu_count() or 4)
print(f"CPU threads: {torch.get_num_threads()}")

model = torch.jit.load(TRACED_PATH, map_location=device)
model.eval()
print("Model loaded")

# Smoke-test with real inference shape (center=False → 497 frames for 5 s)
with torch.no_grad():
    _dummy = torch.zeros(1, 1, N_MELS, CLIP_SAMPLES // HOP_LENGTH - 3)
    _out   = model(_dummy)
assert _out.shape == (1, NUM_CLASSES), f"Shape mismatch: {_out.shape}"
print(f"Forward pass OK — output shape: {_out.shape}")

In [ ]:
# ── TTA helpers ────────────────────────────────────────────────────────────────
def _time_shift(spec: torch.Tensor, n: int) -> torch.Tensor:
    return torch.roll(spec, n, dims=-1)

def _freq_shift(spec: torch.Tensor, n: int) -> torch.Tensor:
    return torch.roll(spec, n, dims=-2)

TTA_TRANSFORMS = [
    lambda x: x,
    lambda x: _time_shift(x, +50),
    lambda x: _time_shift(x, -50),
    lambda x: _freq_shift(x, +4),
]


@torch.no_grad()
def predict_batch(model, specs: torch.Tensor) -> torch.Tensor:
    """Run inference (with or without TTA). specs: (W, 1, mels, T) → (W, classes)"""
    if not USE_TTA:
        probs = []
        for i in range(0, len(specs), BATCH_SIZE):
            probs.append(torch.sigmoid(model(specs[i : i + BATCH_SIZE])))
        return torch.cat(probs, dim=0)

    # TTA: average over all views
    view_probs = []
    for tfm in TTA_TRANSFORMS:
        aug = torch.stack([tfm(specs[i]) for i in range(len(specs))])
        probs = []
        for i in range(0, len(aug), BATCH_SIZE):
            probs.append(torch.sigmoid(model(aug[i : i + BATCH_SIZE])))
        view_probs.append(torch.cat(probs, dim=0))
    return torch.stack(view_probs).mean(dim=0)


print(f"TTA helpers ready  (USE_TTA={USE_TTA}, {len(TTA_TRANSFORMS)} views available)")

In [ ]:
# ── Co-occurrence booster ──────────────────────────────────────────────────────
COOCCUR_ALPHA     = 0.15
COOCCUR_THRESHOLD = 0.40
CLASS_TO_IDX      = {c: i for i, c in enumerate(ALL_CLASSES)}


def build_cooccurrence(labels_csv_path: str) -> np.ndarray:
    counts    = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.float32)
    marginals = np.zeros(NUM_CLASSES, dtype=np.float32)
    with open(labels_csv_path) as f:
        for row in csv.DictReader(f):
            labels = [s.strip() for s in str(row["primary_label"]).split(";") if s.strip()]
            idxs   = [CLASS_TO_IDX[l] for l in labels if l in CLASS_TO_IDX]
            for i in idxs:
                marginals[i] += 1
                for j in idxs:
                    counts[i, j] += 1
    mat = counts / np.maximum(marginals, 1)[:, np.newaxis]
    np.fill_diagonal(mat, 0.0)
    print(f"Co-occurrence matrix: {(mat > 0.3).sum()} species pairs with P > 0.3")
    return mat


def cooccur_boost(probs_np: np.ndarray, mat: np.ndarray) -> np.ndarray:
    detected = (probs_np >= COOCCUR_THRESHOLD).astype(np.float32)
    boost    = detected @ mat
    return np.clip(probs_np + COOCCUR_ALPHA * boost * (1 - probs_np), 0.0, 1.0)


labels_csv = os.path.join(COMP_DIR, "train_soundscapes_labels.csv")
if os.path.isfile(labels_csv):
    COOCCUR_MAT = build_cooccurrence(labels_csv)
    USE_COOCCUR = True
else:
    COOCCUR_MAT = None
    USE_COOCCUR = False
    print("train_soundscapes_labels.csv not found — co-occurrence disabled")

In [ ]:
# ── Discover test soundscapes + runtime estimate ───────────────────────────────
test_files = sorted(Path(COMP_DIR, "test_soundscapes").glob("*.ogg"))
print(f"Test files: {len(test_files)}")

if test_files:
    try:
        import soundfile as sf
        dur_s = sf.info(str(test_files[0])).duration
    except Exception:
        import librosa
        dur_s = librosa.get_duration(path=str(test_files[0]))

    wpf        = math.floor(dur_s / 5)
    total_wins = len(test_files) * wpf
    tta_mult   = len(TTA_TRANSFORMS) if USE_TTA else 1

    # Timing: CNN-only ensembles ~100 ms/window; any model with AST ~550 ms/window
    has_ast    = "v2" in TRACED_FILENAME or "ast" in TRACED_FILENAME.lower()
    ms_per_win = 550 if has_ast else 100
    if has_ast:
        print("  ⚠  AST model detected — this will likely exceed the 90-min budget.")
        print("     Switch TRACED_FILENAME to 'ensemble_baseline_panns.pt'.")

    est_min = (total_wins * ms_per_win * tta_mult / 1000 / 60) + (len(test_files) * 0.15 / 60)
    print(f"  ~{wpf} windows/file  ·  {total_wins:,} total windows  ·  TTA×{tta_mult}")
    print(f"  Estimated runtime: {est_min:.1f} min  (budget: 90 min)")
    if est_min > 75 and not has_ast:
        print("  ⚠  Tight — consider disabling TTA or reducing batch complexity.")

In [ ]:
# ── Inference ──────────────────────────────────────────────────────────────────
all_rows = []
t_start  = time.time()

with torch.no_grad():
    for fi, path in enumerate(test_files):
        end_secs, specs = file_to_specs(str(path))
        if not end_secs:
            continue

        probs_np = predict_batch(model, specs).numpy()  # (W, NUM_CLASSES)

        if USE_COOCCUR:
            probs_np = cooccur_boost(probs_np, COOCCUR_MAT)

        stem = path.stem
        for end_sec, row_probs in zip(end_secs, probs_np):
            row = {"row_id": f"{stem}_{end_sec}"}
            row.update(zip(ALL_CLASSES, row_probs.tolist()))
            all_rows.append(row)

        if (fi + 1) % 50 == 0 or (fi + 1) == len(test_files):
            elapsed   = time.time() - t_start
            remaining = elapsed / (fi + 1) * (len(test_files) - fi - 1)
            print(f"  {fi+1}/{len(test_files)} files | {len(all_rows)} windows | "
                  f"elapsed {elapsed/60:.1f}m | ETA {remaining/60:.1f}m")

print(f"\nDone — {len(all_rows)} windows in {(time.time()-t_start)/60:.1f} min")

In [ ]:
# ── Save & validate submission.csv ────────────────────────────────────────────
submission = pd.DataFrame(all_rows, columns=["row_id"] + ALL_CLASSES)
submission.to_csv(OUT_CSV, index=False)
print(f"Saved → {OUT_CSV}")
print(f"Shape:  {submission.shape}")

sample = pd.read_csv(os.path.join(COMP_DIR, "sample_submission.csv"))
cols_ok = list(submission.columns) == list(sample.columns)
print(f"Columns match sample_submission: {cols_ok}")
if not cols_ok:
    missing = set(sample.columns) - set(submission.columns)
    extra   = set(submission.columns) - set(sample.columns)
    if missing: print(f"  Missing: {missing}")
    if extra:   print(f"  Extra:   {extra}")

if len(all_rows) == 0:
    print("⚠  No predictions — test_soundscapes/ is empty.")
    print("   This is normal when running interactively; test files appear only during commit.")
else:
    prob_vals = submission[ALL_CLASSES].values
    print(f"Prob range: [{prob_vals.min():.4f}, {prob_vals.max():.4f}]")
    print(f"Mean prob:  {prob_vals.mean():.4f}")